In [ ]:
# ============================================================
# Conformer ASR Fine-Tuning on Mozilla Common Voice (en-AU)
# Uses: NVIDIA NeMo - Conformer-CTC-Small (pretrained)
# ============================================================

# ── 1. Install dependencies ──────────────────────────────────
!pip install "numpy>=1.26,<2.0" --quiet
!pip install nemo_toolkit['asr'] --quiet
!pip install wget --quiet

import os, random, json, math, warnings
import pandas as pd
import numpy as np
import torch
import librosa
import soundfile as sf
from pathlib import Path
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 93.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pyannote-metrics 4.0.0 requires numpy>=2.2.2, but you have numpy 1.26.4 which is incompatible.
pyannote-core 6.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.18.1 requires numba<0.63,>=0.60, but you have numba 0.64.0 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 w

In [2]:
# ── 2. Config ────────────────────────────────────────────────
CSV_PATH   = "/kaggle/input/datasets/eddiehoogewerf/mozilla-commonvoice/commonvoice-v24_en-AU/commonvoice-v24_en-AU.csv"
CLIPS_DIR  = "/kaggle/input/datasets/eddiehoogewerf/mozilla-commonvoice/commonvoice-v24_en-AU/audio_files"
WORK_DIR   = "/kaggle/working"
WAV_DIR    = f"{WORK_DIR}/wavs"          # converted WAVs live here
MANIFEST_DIR = f"{WORK_DIR}/manifests"

os.makedirs(WAV_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

SAMPLE_RATE  = 16_000
MAX_DURATION = 15.0          # seconds – drop longer clips
MIN_DURATION = 0.5
MAX_SAMPLES  = 15_000        # cap total samples to fit in 30 h
BATCH_SIZE   = 16
EPOCHS       = 30            # NeMo will early-stop if val loss plateaus
LR           = 1e-4



In [3]:
# ── 3. Load & clean CSV ──────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df = df[["path", "sentence"]].dropna().reset_index(drop=True)

# Normalize text: lowercase, strip punctuation except apostrophes
import re
def clean_text(t):
    t = t.lower().strip()
    t = re.sub(r"[^a-z0-9\s\']", "", t)
    t = re.sub(r"\s+", " ", t)
    return t

df["sentence"] = df["sentence"].apply(clean_text)
df = df[df["sentence"].str.len() > 0].reset_index(drop=True)

# Optional: subsample to fit GPU budget
if len(df) > MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES, random_state=42).reset_index(drop=True)

print(f"Total samples: {len(df)}")



Total samples: 15000


In [4]:
# ── 4. Convert MP3 → 16 kHz mono WAV & filter by duration ───
def convert_and_check(row):
    src = os.path.join(CLIPS_DIR, row["path"])
    dst = os.path.join(WAV_DIR, Path(row["path"]).stem + ".wav")
    if not os.path.exists(dst):
        try:
            y, sr = librosa.load(src, sr=SAMPLE_RATE, mono=True)
            sf.write(dst, y, SAMPLE_RATE)
        except Exception:
            return None, None
    try:
        info = sf.info(dst)
        dur  = info.duration
    except Exception:
        return None, None
    if not (MIN_DURATION <= dur <= MAX_DURATION):
        return None, None
    return dst, dur

print("Converting audio (this takes a while)…")
paths, durations = [], []
for _, row in df.iterrows():
    p, d = convert_and_check(row)
    paths.append(p)
    durations.append(d)

df["wav_path"] = paths
df["duration"] = durations
df = df.dropna(subset=["wav_path"]).reset_index(drop=True)
print(f"After duration filter: {len(df)} samples")





Converting audio (this takes a while)…
After duration filter: 15000 samples


In [5]:
# ── 5. Train / Val / Test split ──────────────────────────────
train_df, tmp_df = train_test_split(df, test_size=0.15, random_state=42)
val_df,  test_df = train_test_split(tmp_df, test_size=0.5,  random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ── 6. Write NeMo manifests (JSON-lines) ────────────────────
def write_manifest(df, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            entry = {
                "audio_filepath": row["wav_path"],
                "duration":       float(row["duration"]),
                "text":           row["sentence"],
            }
            f.write(json.dumps(entry) + "\n")

TRAIN_MANIFEST = f"{MANIFEST_DIR}/train.json"
VAL_MANIFEST   = f"{MANIFEST_DIR}/val.json"
TEST_MANIFEST  = f"{MANIFEST_DIR}/test.json"

write_manifest(train_df, TRAIN_MANIFEST)
write_manifest(val_df,   VAL_MANIFEST)
write_manifest(test_df,  TEST_MANIFEST)
print("Manifests written.")

Train: 12750 | Val: 1125 | Test: 1125
Manifests written.


In [6]:
# ── 7. Load pretrained Conformer-CTC model ───────────────────
import nemo.collections.asr as nemo_asr

MODEL_NAME = "stt_en_conformer_ctc_small"   # ~30M params, fast to fine-tune
# Alternatives (heavier, better accuracy):
#   "stt_en_conformer_ctc_medium"  ~120M params
#   "stt_en_conformer_ctc_large"   ~120M params

asr_model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(MODEL_NAME)
print(f"Loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in asr_model.parameters()):,}")



[NeMo W 2026-03-02 22:47:32 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


[NeMo I 2026-03-02 22:47:35 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/stt_en_conformer_ctc_small/versions/1.6.0/files/stt_en_conformer_ctc_small.nemo to /root/.cache/torch/NeMo/NeMo_2.7.0/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo
[NeMo I 2026-03-02 22:47:35 common:939] Instantiating model from pre-trained checkpoint
[NeMo I 2026-03-02 22:47:36 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-02 22:47:36 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/NeMo_ASR_SET/English/v2.0/train/tarred_audio_manifest.json
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20.0
    min_duration: 0.1
    shuffle_n: 2048
    is_tarred: true
    tarred_audio_filepaths: /data/NeMo_ASR_SET/English/v2.0/train/audio__OP_0..4095_CL_.tar
    
[NeMo W 2026-03-02 22:47:36 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/ASR/LibriSpeech/librispeech_withs

[NeMo I 2026-03-02 22:47:37 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.0/stt_en_conformer_ctc_small/5d2d8e5b2b5adb8f5091363c6ba19c55/stt_en_conformer_ctc_small.nemo.
Loaded: stt_en_conformer_ctc_small
Parameters: 13,154,033


In [7]:
# ── 8. Update data config to point at our manifests ─────────
asr_model.setup_training_data(train_data_config={
    "manifest_filepath": TRAIN_MANIFEST,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
    "shuffle": True,
    "num_workers": 4,
    "pin_memory": True,
    "max_duration": MAX_DURATION,
    "trim_silence": True,
})

asr_model.setup_validation_data(val_data_config={
    "manifest_filepath": VAL_MANIFEST,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
    "shuffle": False,
    "num_workers": 4,
    "pin_memory": True,
    "max_duration": MAX_DURATION,
})

asr_model.setup_test_data(test_data_config={
    "manifest_filepath": TEST_MANIFEST,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
    "shuffle": False,
    "num_workers": 4,
    "pin_memory": True,
    "max_duration": MAX_DURATION,
})



[NeMo I 2026-03-02 22:47:37 collections:201] Dataset loaded with 12750 files totalling 17.79 hours
[NeMo I 2026-03-02 22:47:37 collections:202] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-03-02 22:47:37 collections:201] Dataset loaded with 1125 files totalling 1.57 hours
[NeMo I 2026-03-02 22:47:37 collections:202] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-03-02 22:47:38 collections:201] Dataset loaded with 1125 files totalling 1.56 hours
[NeMo I 2026-03-02 22:47:38 collections:202] 0 files were filtered totalling 0.00 hours


In [8]:
# ── 9. Optimizer & scheduler ─────────────────────────────────
from omegaconf import OmegaConf, open_dict

with open_dict(asr_model.cfg):
    asr_model.cfg.optim = OmegaConf.create({
        "name": "adamw",
        "lr": LR,
        "betas": [0.9, 0.98],
        "weight_decay": 1e-3,
        "sched": {
            "name": "CosineAnnealing",
            "warmup_steps": 500,
            "warmup_ratio": None,
            "min_lr": 1e-6,
        }
    })

asr_model.setup_optimization()

# ── 10. Freeze encoder for first few epochs (optional speedup)
# Uncomment if you want to train only the decoder first:
# asr_model.encoder.freeze()
# print("Encoder frozen – training decoder only")



[NeMo W 2026-03-02 22:47:38 modelPT:708] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2026-03-02 22:47:38 modelPT:830] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: [0.9, 0.98]
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 0.0001
        maximize: False
        weight_decay: 0.001
    )


[NeMo W 2026-03-02 22:47:38 lr_scheduler:975] Neither `max_steps` nor `iters_per_batch` were provided to `optim.sched`, cannot compute effective `max_steps` !
    Scheduler will not be instantiated !


(AdamW (
 Parameter Group 0
     amsgrad: False
     betas: [0.9, 0.98]
     capturable: False
     decoupled_weight_decay: True
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0001
     maximize: False
     weight_decay: 0.001
 ),
 None)

In [9]:
# ── FALLBACK: Pure NeMo 2.7 training loop (no PTL Trainer) ──────────

from nemo.core.config import hydra_runner
from omegaconf import OmegaConf, DictConfig
import torch

def train_loop(model, train_manifest, val_manifest, epochs=EPOCHS, lr=LR):
    model = model.cuda()
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6
    )

    # Setup data
    model.setup_training_data({"manifest_filepath": train_manifest,
                                "sample_rate": SAMPLE_RATE,
                                "batch_size": BATCH_SIZE,
                                "shuffle": True,
                                "num_workers": 4,
                                "pin_memory": True,
                                "max_duration": MAX_DURATION,
                                "trim_silence": True})
    model.setup_validation_data({"manifest_filepath": val_manifest,
                                  "sample_rate": SAMPLE_RATE,
                                  "batch_size": BATCH_SIZE,
                                  "shuffle": False,
                                  "num_workers": 4,
                                  "pin_memory": True,
                                  "max_duration": MAX_DURATION})

    train_loader = model._train_dl
    val_loader   = model._validation_dl

    scaler = torch.cuda.amp.GradScaler()
    best_val_loss = float("inf")
    patience_counter = 0
    PATIENCE = 5

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        total_loss, steps = 0.0, 0
        for batch in train_loader:
            batch = [x.cuda() if isinstance(x, torch.Tensor) else x for x in batch]
            signal, signal_len, transcript, transcript_len = batch

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                log_probs, encoded_len, greedy_preds = model(
                    input_signal=signal, input_signal_length=signal_len
                )
                loss = model.loss(
                    log_probs=log_probs,
                    targets=transcript,
                    input_lengths=encoded_len,
                    target_lengths=transcript_len,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            steps += 1

        avg_train_loss = total_loss / steps
        scheduler.step()

        # ── Validate ──
        model.eval()
        val_loss, val_steps = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = [x.cuda() if isinstance(x, torch.Tensor) else x for x in batch]
                signal, signal_len, transcript, transcript_len = batch
                with torch.cuda.amp.autocast():
                    log_probs, encoded_len, greedy_preds = model(
                        input_signal=signal, input_signal_length=signal_len
                    )
                    loss = model.loss(
                        log_probs=log_probs,
                        targets=transcript,
                        input_lengths=encoded_len,
                        target_lengths=transcript_len,
                    )
                val_loss += loss.item()
                val_steps += 1

        avg_val_loss = val_loss / val_steps
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # ── Checkpoint ──
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            model.save_to(f"{WORK_DIR}/checkpoints/best_model.nemo")
            print(f"  ✓ Saved best model (val_loss={best_val_loss:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    return model

print("\n=== Starting fine-tuning (NeMo 2.7 native loop) ===")
asr_model = train_loop(asr_model, TRAIN_MANIFEST, VAL_MANIFEST)


=== Starting fine-tuning (NeMo 2.7 native loop) ===
[NeMo I 2026-03-02 22:47:38 collections:201] Dataset loaded with 12750 files totalling 17.79 hours
[NeMo I 2026-03-02 22:47:38 collections:202] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-03-02 22:47:38 collections:201] Dataset loaded with 1125 files totalling 1.57 hours
[NeMo I 2026-03-02 22:47:38 collections:202] 0 files were filtered totalling 0.00 hours
Epoch 1/30 | Train Loss: 16.8576 | Val Loss: 6.7843
  ✓ Saved best model (val_loss=6.7843)
Epoch 2/30 | Train Loss: 15.5211 | Val Loss: 6.6672
  ✓ Saved best model (val_loss=6.6672)
Epoch 3/30 | Train Loss: 14.9541 | Val Loss: 6.6221
  ✓ Saved best model (val_loss=6.6221)
Epoch 4/30 | Train Loss: 14.6155 | Val Loss: 6.6562
Epoch 5/30 | Train Loss: 14.3478 | Val Loss: 6.6455
Epoch 6/30 | Train Loss: 13.9738 | Val Loss: 6.5416
  ✓ Saved best model (val_loss=6.5416)
Epoch 7/30 | Train Loss: 13.7987 | Val Loss: 6.5590
Epoch 8/30 | Train Loss: 13.5175 | Val Loss: 6.5300
  ✓

In [10]:
# ── Load best checkpoint before evaluating ───────────────────
from nemo.collections.asr.models import EncDecCTCModelBPE
from nemo.collections.asr.metrics.wer import word_error_rate
import random

# Load the best model saved during training
best_model = EncDecCTCModelBPE.restore_from(f"{WORK_DIR}/checkpoints/best_model.nemo")
best_model = best_model.cuda()
best_model.eval()

# ── Evaluate on test set ──────────────────────────────────────
test_audio = test_df["wav_path"].tolist()
test_refs  = test_df["sentence"].tolist()

raw_hypotheses = best_model.transcribe(test_audio, batch_size=16)

# NeMo 2.7 returns Hypothesis objects — extract the text
hypotheses = []
for h in raw_hypotheses:
    if hasattr(h, 'text'):
        hypotheses.append(h.text)
    elif hasattr(h, 'y_sequence'):
        hypotheses.append(str(h.y_sequence))
    else:
        hypotheses.append(str(h))

wer = word_error_rate(hypotheses=hypotheses, references=test_refs)
cer = word_error_rate(hypotheses=hypotheses, references=test_refs, use_cer=True)

print(f"\n{'='*40}")
print(f"  Test WER : {wer*100:.2f}%")
print(f"  Test CER : {cer*100:.2f}%")
print(f"{'='*40}")

# Sample predictions
print("\nSample predictions:")
for i in random.sample(range(len(test_refs)), min(10, len(test_refs))):
    print(f"  REF : {test_refs[i]}")
    print(f"  HYP : {hypotheses[i]}")
    print()

# Save final model
best_model.save_to(f"{WORK_DIR}/conformer_finetuned_final.nemo")
print(f"\nFinal model saved.")

[NeMo I 2026-03-02 23:28:34 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-03-02 23:28:35 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/manifests/train.json
    sample_rate: 16000
    batch_size: 16
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 15.0
    trim_silence: true
    
[NeMo W 2026-03-02 23:28:35 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/manifests/val.json
    sample_rate: 16000
    batch_size: 16
    shuffle: false
    num_workers: 4
    pin_memory: true
    max_duration: 15.0
    
[NeMo W 2026-03-02 23:28:35 modelPT:202] Please call the ModelPT.setup_test_data() or ModelPT.s

[NeMo I 2026-03-02 23:28:35 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /kaggle/working/checkpoints/best_model.nemo.


[NeMo W 2026-03-02 23:28:35 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-02 23:28:35 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 71it [00:35,  1.97it/s]



  Test WER : 10.47%
  Test CER : 3.81%

Sample predictions:
  REF : your good health
  HYP : your good health

  REF : the minidoka irrigation project shares its name with minidoka county
  HYP : the mideka irrigation project shares its name with midega county

  REF : they include officers sergeants and other ranks
  HYP : they include offices sergeants and other ranks

  REF : modern sacks are often made from manmade products such as polypropylene
  HYP : modern sacks are often made from manmade products such as polypropline

  REF : this refusal to condemn violence brought the agreements to an end
  HYP : this refusal to condemn violence brought the agreements to an end

  REF : test tubes are usually placed in specialpurpose racks
  HYP : test tubes are usually placed in special purpose racks

  REF : they are narrow and oblong in shape
  HYP : they are narrow and oblong in shape

  REF : there was no justification for it
  HYP : there was no justification for it

  REF : children